Import 

In [163]:
import cv2
import numpy as np
from PIL import Image

image_path = "Images/dr.png"

image = cv2.imread(image_path)


In [164]:
def display(image, label):
    resize = cv2.resize(image, None,fx=0.5,fy=0.5,interpolation=cv2.INTER_AREA);
    
    cv2.imshow(label,image);
    #cv2.imshow(label,resize);

    cv2.waitKey(0);
    cv2.destroyAllWindows();

In [165]:
#display(image,"image")

Invert

In [166]:
def invert(image):
    inv = cv2.bitwise_not(image)
    return inv

inverted = invert(image);
#display(invert(image),"Inverted")

Grayscale

In [167]:
def grayScale(image):
    gray = cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    return gray;

gray = grayScale(inverted);

#display(gray,"Gray")

Threshold

In [168]:
def thresholdBin(image,low=0,high=255):
    thr, bw = cv2.threshold(image,low,high,cv2.THRESH_BINARY);
    return bw;

threshold = thresholdBin(gray,50,255);

#display(threshold, "Threshold");

Thin/Thick 

In [169]:
def thick(image,iteration=1,x=1,y=1):
    kernel = np.ones((x,y), np.uint8);
    dilate = cv2.dilate(image,kernel,iteration);
    return dilate;

In [170]:
def thin(image,iteration=1,x=1,y=1):
    kernel = np.ones((x,y), np.uint8);
    erode = cv2.erode(image,kernel,iteration);
    return erode;

Remove Noise

In [171]:
def removeNoise(image,dilate=False,dIteration=0,erode=False,eIteration=0):
    dKernel = np.ones((1,1),np.uint8)
    eKernel = np.ones((1,1),np.uint8)
    mKernel = np.ones((1,1),np.uint8)
    if dilate == True:
        image = cv2.dilate(image,dKernel,dIteration);

    if erode == True:
        image = cv2.erode(image,eKernel,eIteration);

    image = cv2.morphologyEx(image,cv2.MORPH_CLOSE,mKernel);
    image = cv2.medianBlur(image,3);

    return image;

In [172]:
noise_remove = removeNoise(threshold, dilate=True,dIteration=10,erode=False,eIteration=1);

#display(noise_remove, "Noise Reduction")

Make Fonts Thinner

In [173]:
thinner = thin(noise_remove,iteration=1,x=2,y=2)
thicker = thick(thinner,iteration=1,x=2,y=2)
display(thicker,"Thinner Then Thicker")

Get Skew Angle

In [174]:
def getSkewAngle(image) -> float:
    copy = image.copy()

    gray = cv2.cvtColor(copy, cv2.COLOR_BGR2GRAY);
    gaussian = cv2.GaussianBlur(gray,(11,11),0)
    threshold = cv2.threshold(gaussian,0,255,cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT,(30,5))
    dilate = cv2.dilate(threshold,kernel,iterations=2)

    contours, hierarchy = cv2.findContours(dilate,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours,key = cv2.contourArea, reverse=True)
    for c in contours:
        rect = cv2.boundingRect(c)
        x,y,w,h = rect
        cv2.rectangle(copy,(x,y),(x+w,y+h),(0,255,0),2)

    large_contour = contours[0]
    minAreaRect = cv2.minAreaRect(large_contour)

    cv2.imwrite("temp/c.jpg",copy);
    angle =  minAreaRect[-1]

    if angle < -45:
        angle = 90 + angle
    
    return -1.0*angle



In [175]:
def rotate(image,angle:float):
    copy = image.copy()
    (h,w) = copy.shape[:2]
    center = (w//2,h//2)

    M = cv2.getRotationMatrix2D(center,angle,1.0)

    copy = cv2.warpAffine(copy,M,(w,h),flags=cv2.INTER_CUBIC,borderMode = cv2.BORDER_REPLICATE)

    return copy

In [176]:
def deskew(image):
    angle = getSkewAngle(image);
    return rotate(image,-1.0*angle)

In [180]:
fix = deskew(image)

display(fix,"Fixed")

Remove Borders

In [178]:
def remove_borders(image):
    contours, heiarchy = cv2.findContours(image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cntsSorted = sorted(contours, key=lambda x:cv2.contourArea(x))
    cnt = cntsSorted[-1]
    x, y, w, h = cv2.boundingRect(cnt)
    crop = image[y:y+h, x:x+w]
    return (crop)

In [179]:
no_borders = remove_borders(noise_remove)
cv2.imwrite("temp/no_borders.jpg", no_borders)

True

In [181]:
import pytesseract
from PIL import Image

In [183]:
ocr = pytesseract.image_to_string(fix)

print(ocr)

on ©

ster morning in the year 19%, I took my six-year-old

son by the hand and began walking from my home town toward the

valleys and forests of the Carpathien mountains. For nearly
eight months we lived in barns, attics and makeshift cabins. with
the generous help of an unusually courageous man, we managed to
survive Europe's greatest fit of madness. Those who walked in

the opposite direction on that Easter day were less fortunate.
They were taken in trainloads to places whose once obscure nanes
are now, and forever will be, synonymous with terror, evil and
death. What follows is our story of survival told to the best

of my ability, in plain, simple language.

In March of 1944 the SS troops took over the internal affairs
of Hungary and proceeded to organize the deportation of the Jews.
fo the Nazis this was a routine assignment; within hours all local
officials were informed of operational plans. The high commend
issued a directive designed to placate Jewish fears and induce
coope